In [1]:
import numpy as np
import pandas as pd

import torch

import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore",category=DeprecationWarning)
    import pytorch_lightning as pl

In [2]:
from rosetta_db import get_sqlite_dbcon
db_con = get_sqlite_dbcon(parent="2D")
df = pd.read_sql("select r.*, d.* from relaxed_scores as r "
                 "inner join docked_scores as d on d.variant = r.variant;"
                 , db_con)

# We have duplicated column names because they have similar names in the two columns
# now relabel the columns so that the first columns are prepended with r_ for relaxation
# and the second set of columns are prepended with d_ for docking

# description should occur twice in the database columns. Find out where
desciption_idxs = np.where(df.columns == "description")[0]
assert(len(desciption_idxs) == 2)
table_split = desciption_idxs[0] + 1

df.columns = ["r_" + c for c in df.columns[:table_split]] \
           + ["d_" + c for c in df.columns[table_split:-1]] \
           + [df.columns[-1]] # this is "variant", the key for joining so it only appears once

# For relaxed_scores it is always the same structure e.g. SadA_NSLeu_2D_0001_0001 as we only do one structure
# For docked_scores it is the best structure out of nstructs=100. 
# We do not need this information for modeling

columns_to_drop = ["r_description","d_description", "d_variant" ] # which structure was selected. 
columns_to_drop.append("r_dslf_fa13") # from relaxed scores

# columns to delete from docked scores
# These are all constant (std = 0)
columns_to_drop += list(df.describe().loc[:, df.describe().loc['std'] == 0].columns)

print(f"Dropping {len(columns_to_drop)} columns: ", columns_to_drop)

df.drop(columns=columns_to_drop, inplace=True)
df.head()

Dropping 30 columns:  ['r_description', 'd_description', 'd_variant', 'r_dslf_fa13', 'r_dslf_fa13', 'd_angle_constraint', 'd_atom_pair_constraint', 'd_chainbreak', 'd_dihedral_constraint', 'd_dslf_ca_dih', 'd_dslf_cs_ang', 'd_dslf_ss_dih', 'd_dslf_ss_dst', 'd_if_X_angle_constraint', 'd_if_X_atom_pair_constraint', 'd_if_X_chainbreak', 'd_if_X_coordinate_constraint', 'd_if_X_dihedral_constraint', 'd_if_X_dslf_ca_dih', 'd_if_X_dslf_cs_ang', 'd_if_X_dslf_ss_dih', 'd_if_X_dslf_ss_dst', 'd_if_X_fa_dun', 'd_if_X_hbond_lr_bb', 'd_if_X_hbond_sr_bb', 'd_if_X_omega', 'd_if_X_p_aa_pp', 'd_if_X_pro_close', 'd_if_X_ref', 'd_ligand_is_touching_X']


,r_total_score,r_fa_atr,r_fa_dun,r_fa_elec,r_fa_intra_rep,r_fa_intra_sol_xover4,r_fa_rep,r_fa_sol,r_hbond_bb_sc,r_hbond_lr_bb,...,d_if_X_fa_sol,d_if_X_hbond_bb_sc,d_if_X_hbond_sc,d_interface_delta_X,d_omega,d_p_aa_pp,d_pro_close,d_ref,d_total_score_X,variant
0,-854.046,-1547.960,312.281,-531.071,2.815,54.993,179.953,923.799,-55.735,-83.625,...,7.740,-3.733,-2.840,-10.495,48.539,-42.449,1.057,-44.53,-29.0,F240S
1,-844.229,-1562.435,321.579,-535.211,2.884,56.004,184.718,941.687,-56.275,-82.617,...,8.260,-5.314,-0.423,-11.265,50.349,-41.727,1.029,-45.96,-29.0,A187R.V39N.F62K
2,-826.918,-1569.686,312.137,-516.471,3.134,56.338,185.486,918.217,-54.229,-83.145,...,6.567,-1.979,-3.298,-9.093,48.736,-41.179,1.080,-39.69,-29.0,D190I.Q236I.T254S.E134P.E208C.G180R.A69W
3,-848.450,-1563.670,324.802,-541.695,2.829,56.087,182.194,948.365,-58.665,-82.374,...,11.059,-0.351,-9.971,-12.745,48.305,-43.662,1.040,-40.86,-29.0,H3W.V142G.T254P.F150W.A18R.G79Q.A15C.A187T.I248R
4,-859.796,-1556.332,312.891,-534.450,2.852,55.373,180.526,928.443,-57.656,-83.463,...,6.634,-0.322,-3.433,-7.656,48.245,-42.532,1.132,-47.13,-30.0,F97L.H210R


In [3]:
# drop the first r_energy term (r_total_score) and sum up the other r_ energy terms
# we should get something very close to the r_total_score
print(f"Sum of r_ energy terms for first variant : {df.loc[0, df.columns.str.startswith('r_')][1:].sum():.4f}")
print(f"    Total energy terms for first variant : {df.loc[0, 'r_total_score']:.4f}")

Sum of r_ energy terms for first variant : -854.0440
    Total energy terms for first variant : -854.0460
